# CS224N · Lecture 18 — NLP, Linguistics, and Philosophy

**Slides:** [cs224n-spr2024-lecture18-nlp-linguistics-philosophy.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture18-nlp-linguistics-philosophy.pdf) (Christopher Manning)  
**Companion:** the Lecture 19 notebook (Hashimoto, *Open problems*) covers the open problems in more depth; this one focuses on Manning's themes: symbolic vs. neural systems, meaning, and the future risks of AI.

| § | Topic | You will implement / measure |
|---|---|---|
| 1 | Major ideas of CS224N | — (a map back to the notebooks) |
| 2 | Open problems: generalization ("shortcuts"), analysis, multilinguality, evaluation, domains | A fixed-depth "shortcut" that breaks beyond its horizon |
| 3 | Where are we with LLMs? | — |
| 4 | **Symbolic AI vs. cybernetics** | Rosenblatt's perceptron: the convergence bound, and XOR |
| 5 | **Meaning**: model-theoretic vs. distributional semantics; compositionality; Montague | A compositional semantic interpreter; semantic parsing to SQL |
| 6 | Meaning as use (Wittgenstein); "meaning is gradient" (*shehnai*) | Meaning from contexts, growing with each context |
| 7 | The future risks of AI | 🔎 LLM watermarking: generation and statistical detection |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math, random, collections, re, sqlite3, itertools

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)
random.seed(0)

def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True); e = np.exp(z); return e / e.sum(axis=axis, keepdims=True)

---
## 1. Major ideas of CS224N

| Idea | What it means | Where we built it |
|---|---|---|
| **1. Dense representations and distributional semantics** | Predict neighboring words using a word-vector representation | L1–2 (word2vec, GloVe) |
| **2. Depth and neural networks** | The challenges of training large, deep nets; tricks like residual connections | L3 (backprop), L6 (residuals), L8, L12 |
| **3. Sequence models and computational efficiency** | Vanishing gradients and information propagation; the parallelization and dependency benefits of Transformers | L5–8, L16, L18 (efficiency) |
| **4. Language modeling and pretraining** | LM as a "universal" pretraining task; careful, smart scaling to internet-wide data | L5, L9–10 |
| … and more | Benchmarking, reasoning, responsible NLP, brain–computer interfaces | L11, L14, L17, L13 |

---
## 2. Open problems (a quick and incomplete overview)

**Generalization · Analysis and understanding · Multilinguality · Evaluation · Domains and modalities · Fairness and social impact.**

### 2.1 How much do models really generalize?
Even modern LLMs seem to leverage **surface cues**: are we just finding better shortcuts? Liu et al. (ICLR 2023, *Transformers Learn Shortcuts to Automata*) show that a shallow Transformer can simulate a finite automaton over a sequence of length $T$ with far fewer than $T$ layers, by learning **shortcut** solutions (e.g., parallel prefix computations of depth $\sim\log T$) instead of the step-by-step recurrence. Shortcuts fit the training distribution but can be **brittle** out of distribution (e.g., at longer lengths), and on limited data **a Transformer can be worse than an LSTM** at learning an automaton's output.

A cartoon of why: the **recurrent** solution to parity (running XOR) works for any length. A **parallel shortcut** with a fixed number of "layers" (doubling steps of a prefix scan, L18's parallel-scan trick) only combines information across $2^{\text{layers}}$ positions, so it's perfect up to that horizon and wrong beyond it:

In [ ]:
def parity_recurrent(bits):
    out, s = [], 0
    for b in bits:
        s ^= b; out.append(s)                       # one step per token: works for any length
    return np.array(out)

def parity_shortcut(bits, layers):
    x = np.array(bits).copy()                       # Hillis-Steele prefix XOR with a FIXED number of doubling steps
    for l in range(layers):
        shift = 2 ** l
        x = x ^ np.r_[np.zeros(shift, int), x[:-shift]]
    return x

r = np.random.default_rng(0)
layers = 4                                          # enough for sequences up to 2^4 = 16 tokens
for T in [8, 16, 32, 64]:
    accs = []
    for _ in range(500):
        bits = r.integers(0, 2, T)
        accs.append(np.mean(parity_shortcut(bits, layers) == parity_recurrent(bits)))
    print(f"length {T:3d}: shortcut with {layers} 'layers' gets {np.mean(accs):.1%} of prefix parities right")

Within its horizon, the shortcut is perfect (and parallel); beyond it, it degrades toward chance. Nothing in the training data at length ≤ 16 tells the model which solution generalizes. **What solution a model learns, not just its in-distribution accuracy, determines how it generalizes.**

### 2.2 Other open problems from the slides (more in the L19 notebook)
* **What's going on inside neural nets? (Analysis)** We summarize models with one or a handful of accuracy numbers. What do they learn? Why do they succeed and fail? Old results already showed **interpretable latent units** (Karpathy et al. 2016: individual LSTM cells that track quotes or line length; L5 §7.1 probed one). Many more recent results come from **mechanistic interpretability** and **causal abstraction**.
* **Multilingual:** everyone else is worse off than English speakers: significant gaps between high-resource languages (English) and others (e.g., Telugu), even for the best models. Most languages don't have machine-readable written text, many may become extinct, and there's little for-profit motive to serve them: a vicious feedback loop (Adda et al. 2016).
* **Evaluation:** benchmarks drive progress (L11); **pretraining makes it increasingly difficult to build hidden test sets**.
* **Domains:** **biomedical/clinical NLP** has enormous potential (and risks): note-taking, QA, "curbside consults". **Legal NLP** might help address the **"justice gap"**, but systems must understand complex jargon and be reliable, and at present they **hallucinate**.
* **Social aspects:** culture and religion, social norms, under-represented groups (L17).

---
## 3. Where are we with LLMs?

* GPT-4 can write a passable **sonnet** about almost anything on demand.
* **Consultants using GPT-4 outperform non-users** (Dell'Acqua et al. 2023, with Boston Consulting Group): on average they finished **12.2% more tasks**, completed tasks **25.1% more quickly**, and produced **40% higher-quality** results. Use of an LLM **particularly improved the performance of lower-performing humans**. Results vary depending on the task (on a task outside the model's capabilities, users who relied on it did *worse*).
* **Can GPT-4 write fiction that matches the quality of New Yorker fiction? Good news! No!** In a 2023 study (Chakrabarty et al., arXiv:2309.14556), expert-judged LLM stories passed **3–10× fewer** creativity tests than professional stories.
* The *Financial Times* Lex column (Nov 2023), "Generative AI: hypely intelligent", argued that expectations were running ahead of the limitations: models **predict rather than comprehend**, struggle outside the envelope of their training data, struggle to identify their own mistakes, and real-world productivity gains are patchy (e.g., an NBER study of AI assistance for customer-support agents found a **14%** productivity gain, concentrated among **new and low-skilled** workers, with little improvement for experienced ones).

---
## 4. Symbolic AI vs. "cybernetics"

**Stanford, the home of "Symbolic Systems."** *Symbolic systems* studies systems of meaningful symbols that represent the world about us (human languages, logics, programming languages) and the systems that work with these symbols (brains, computers, complex social systems). While cognitive science focuses on the mind and intelligence as naturally occurring phenomena, Symbolic Systems gives equal focus to human-constructed systems that use symbols to communicate and represent information (Jon Barwise, 1942–2000).

**Two visions of artificial intelligence:** **(symbolic) AI** (logic, rules, search) and **cybernetics** (learning, feedback, neural networks). **Frank Rosenblatt** built the **Mark I Perceptron** (1958 proposal; photographed with it in 1960). And early AI hype! The *New York Times*, July 8, 1958: *"NEW NAVY DEVICE LEARNS BY DOING"*, reporting that the Navy expected the computer to be able to *"walk, talk, see, write, reproduce itself and be conscious of its existence."* (It had learned to tell left from right after fifty attempts.)

### 4.1 The perceptron, from scratch
Rosenblatt's learning rule: for each misclassified example $(x, y)$ with $y\in\{-1, +1\}$, update $w \leftarrow w + yx$. **Novikoff's theorem (1962):** if the data are linearly separable with margin $\gamma$ (some unit vector $w^*$ has $y\,w^{*\top}x\geq\gamma$ for all examples) and $\|x\|\leq R$, the perceptron makes **at most $(R/\gamma)^2$ mistakes**. And if the data are *not* linearly separable, like **XOR**, it never converges: the limitation Minsky & Papert (1969) made famous, which a hidden layer (L2–3) removes.

In [ ]:
def perceptron(X, y, max_epochs=1000):
    w = np.zeros(X.shape[1]); mistakes = 0
    for epoch in range(max_epochs):
        errors = 0
        for xi, yi in zip(X, y):
            if yi * (w @ xi) <= 0:
                w += yi * xi; mistakes += 1; errors += 1
        if errors == 0:
            return w, mistakes, epoch + 1
    return w, mistakes, None

r = np.random.default_rng(0)
w_star = np.array([1.0, -2.0, 0.5]); w_star /= np.linalg.norm(w_star)
Xs = np.c_[r.normal(size=(300, 2)), np.ones(300)]
margins = Xs @ w_star; keep = np.abs(margins) > 0.3                     # enforce a margin
Xs, ys = Xs[keep], np.sign(margins[keep])
R, gamma = np.linalg.norm(Xs, axis=1).max(), np.min(ys * (Xs @ w_star))
w, mistakes, epochs = perceptron(Xs, ys)
print(f"separable data: converged in {epochs} epochs with {mistakes} mistakes;  Novikoff bound (R/gamma)^2 = {(R / gamma) ** 2:.0f}")

X_xor = np.array([[0, 0, 1], [0, 1, 1], [1, 0, 1], [1, 1, 1]], float); y_xor = np.array([-1, 1, 1, -1])
_, mistakes, epochs = perceptron(X_xor, y_xor, max_epochs=1000)
print(f"XOR: converged? {epochs is not None}  ({mistakes} mistakes in 1000 epochs: it cycles forever)")

### 4.2 Symbolic systems versus their processors
* **Language is the symbolic system *par excellence*;** we should study and make use of its symbolic structure.
* This does **not** show that the main processor of these symbols, **the human brain, is implemented as a physical symbol system**.
* We **need not design NLP systems as physical symbol systems**. The brain is more like a neural network model; **neural models scale better and can capture the world represented by symbols.**

**Linguistics: tools for empirical language science.** Linguistics gives us questions, concepts, and distinctions for examining languages, language acquisition, and processing, and these tools are just as useful for studying **computer-generated language** and computer language acquisition and processing.

**Linguistic ideas are central to the future of AI.** Fundamental concepts of linguistics are increasingly central to the research program of deep-learning ("gradient-based") AI: **compositionality / factorization, systematic generalization, stable meanings for symbols, manipulating reference.** These concepts are key to going from insect-level intelligence to something like human intelligence. The particular distinctions of particular linguistic theories probably aren't the right thing to focus on in 2020s NLP, but **most of our broad understanding of linguistics is right**, and it's the right tool for the goals, analysis, and evaluation of NLP systems: sentence structure, discourse structure, natural language inference, hyperbole, translationese, prosody, morphology, indirect speech acts, bridging anaphora, metaphor, reference, presupposition, stance, style, coarticulation, tone, …

**Language and thought: von Humboldt (1767–1835).** Human language must *"make infinite use of finite means"* (Chomsky promotes this as the recursive nature of language, in his "Cartesian Linguistics" view). Manning's reading of von Humboldt: language is not merely an outward manifestation of thought for communication; it is **not a product (*Ergon*) but an activity (*Energeia*)**; he effectively distinguishes "system 1" cognition from "system 2" thinking, and system-2 thought requires **the fruitful extension of the mind through the symbols of language**: language is *"the necessary foundation for the progress of the human mind."* (See also Daniel Dennett, *From Bacteria to Bach and Back*.)

---
## 5. Meaning in linguistics and NLP

**What kind of semantics should we use for language?**
* **Model-theoretic semantics:** the meaning of words is their **denotation** (interpretation) in (a model of) the world.
* **Distributional semantics:** understanding the meaning of a word is understanding the **contexts** in which it occurs (L1).

### 5.1 The traditional view: formal compositional semantics
In an intro logic class, *The red apple is on the table* has the meaning $\text{on}(\iota(\lambda x.\,\text{apple}(x)\wedge\text{red}(x)),\ \iota(\lambda y.\,\text{table}(y)))$, where $\iota$ picks out "the unique thing such that". But how do we get the latter from the former, other than by setting undergrads to work?

**Richard Montague (1930–1971):** *"I reject the contention that an important theoretical difference exists between formal and natural languages"* (1968). Partee (1996) on Montague (1970): *the syntax is an algebra, the semantics is an algebra, and there is a homomorphism mapping elements of the syntactic algebra onto elements of the semantic algebra.* The homomorphism requirement **is** the compositionality requirement.

**Two (related) good linguistic properties:**
* **The principle of compositionality** ("Frege's principle"): *the meaning of a whole is a function (only) of the meanings of its (syntactic) parts and the manner in which these parts are combined.* We should be able to put together pieces of linguistic structure we've seen before to interpret new sentences, including longer ones. Exploiting compositionality can give an **exponential gain** in representational power.
* **Systematic generalization:** if a human or model can interpret a noun phrase in subject position, it should also be able to interpret it in object position (Fodor & Pylyshyn 1988). This supports rapid language acquisition: **children of 2 years 11 months can do this** (Brooks & Tomasello 1999).

**Constructing meaning, 1967–2017:** produce a syntactic structure for the sentence; then construct its meaning by **(i) lexical lookup** followed by **(ii) semantic composition**, using a "rule-to-rule" approach up the tree (e.g., PP → P NP, with meaning $\alpha(\beta)$). Let's actually run it, with a small **model** (a world of entities and facts) and Python functions as λ-terms:

In [ ]:
class PresuppositionFailure(Exception):
    pass

def make_lexicon(world):
    def iota(P):                                    # "the": the UNIQUE entity satisfying P (Russell/Strawson)
        xs = [e for e in world["entities"] if P(e)]
        if len(xs) != 1:
            raise PresuppositionFailure(f"'the' needs exactly one such entity, found {len(xs)}")
        return xs[0]
    pred = lambda name: (lambda x: x in world[name])
    return {
        "the": lambda P: iota(P),                                            # λP. ι(P)
        "red": lambda P: (lambda x: P(x) and x in world["red"]),             # λP. λx. (P(x) ∧ red(x))
        "apple": pred("apple"), "table": pred("table"), "box": pred("box"),
        "on": lambda y: (lambda x: (x, y) in world["on"]),                   # λy. λx. on(x, y)
        "is": lambda P: P,                                                   # λP. P
    }

# The syntax tree from the slides:  [S [NP The [N' red apple]] [VP is [PP on [NP the table]]]]
tree = ("S", ("NP", "the", ("N'", "red", "apple")), ("VP", "is", ("PP", "on", ("NP", "the", "table"))))

def interpret(node, lex, trace):
    # Rule-to-rule composition: a branching node's meaning is (functor meaning)(argument meaning).
    if isinstance(node, str):
        return lex[node]
    label, *kids = node
    a, b = (interpret(k, lex, trace) for k in kids)
    meaning = b(a) if label == "S" else a(b)          # S: VP(NP); everything else: left(right)
    trace.append((label, meaning if not callable(meaning) else "<function>"))
    return meaning

worlds = {
    "red apple on the table": {"entities": {"a1", "a2", "t1"}, "apple": {"a1", "a2"}, "red": {"a1"}, "table": {"t1"}, "box": set(), "on": {("a1", "t1")}},
    "red apple on the floor": {"entities": {"a1", "t1"}, "apple": {"a1"}, "red": {"a1"}, "table": {"t1"}, "box": set(), "on": set()},
    "two red apples":         {"entities": {"a1", "a2", "t1"}, "apple": {"a1", "a2"}, "red": {"a1", "a2"}, "table": {"t1"}, "box": set(), "on": {("a1", "t1")}},
}
for name, world in worlds.items():
    trace = []
    try:
        print(f"world '{name}': 'The red apple is on the table' is {interpret(tree, make_lexicon(world), trace)}")
    except PresuppositionFailure as e:
        print(f"world '{name}': no truth value (presupposition failure: {e})")

The meaning of the whole is computed **only** from the meanings of the parts and how they combine, and it works in any world. "The" carries a **presupposition** (exactly one such thing exists): with two red apples, the sentence isn't false, it's *infelicitous*, a distinction formal semantics makes precise.

**Systematic generalization, for free:** the same lexicon and rules interpret *the table is on the red apple* (object NP in subject position) without any new machinery:

In [ ]:
tree2 = ("S", ("NP", "the", "table"), ("VP", "is", ("PP", "on", ("NP", "the", ("N'", "red", "apple")))))
w = worlds["red apple on the table"]
print("'The table is on the red apple' is", interpret(tree2, make_lexicon(w), []))
w_flip = dict(w, on={("t1", "a1")})
print("...and in a world where the table IS on the red apple:", interpret(tree2, make_lexicon(w_flip), []))

### 5.2 This was most of natural language understanding, 1967–2017: semantic parsing
Map a question to an executable logical form or database query. The slides' example: *How many red cars in Palo Alto does Kathy like?*
```sql
select count(*) from Likes, Cars, Locations, Reds where
  Cars.obj = Likes.liked AND Likes.liker = 'Kathy' AND Reds.obj = Likes.liked
  AND Locations.place = 'Palo Alto' AND Locations.obj = Likes.liked
```
Its zenith: symbolic ML **semantic parsing** (Zettlemoyer & Collins 2005; Artzi & Zettlemoyer 2013; Liang, Jordan & Klein 2013). Let's run the slide's query on a toy database, and compare it with composing the meaning from the question's parts:

In [ ]:
db = sqlite3.connect(":memory:")
db.executescript('''
CREATE TABLE Cars(obj TEXT); CREATE TABLE Reds(obj TEXT); CREATE TABLE Locations(obj TEXT, place TEXT);
CREATE TABLE Likes(liker TEXT, liked TEXT);
INSERT INTO Cars VALUES ('c1'), ('c2'), ('c3'), ('c4');
INSERT INTO Reds VALUES ('c1'), ('c2'), ('c4'), ('apple1');
INSERT INTO Locations VALUES ('c1', 'Palo Alto'), ('c2', 'Palo Alto'), ('c3', 'Palo Alto'), ('c4', 'Menlo Park');
INSERT INTO Likes VALUES ('Kathy', 'c1'), ('Kathy', 'c3'), ('Kathy', 'c4'), ('Raj', 'c2');
''')
slide_sql = '''select count(*) from Likes, Cars, Locations, Reds where
  Cars.obj = Likes.liked AND Likes.liker = 'Kathy' AND Reds.obj = Likes.liked
  AND Locations.place = 'Palo Alto' AND Locations.obj = Likes.liked'''
print("the slide's SQL:", db.execute(slide_sql).fetchone()[0])

# Compositional version: each phrase denotes a set, and combining phrases intersects/filters sets.
rows = lambda q: {r_[0] for r_ in db.execute(q)}
cars, reds = rows("select obj from Cars"), rows("select obj from Reds")
in_place = lambda p: rows(f"select obj from Locations where place = '{p}'")
liked_by = lambda who: rows(f"select liked from Likes where liker = '{who}'")
how_many = lambda S: len(S)
print("composed meaning: how_many(red ∩ cars ∩ in(Palo Alto) ∩ liked_by(Kathy)) =",
      how_many(reds & cars & in_place("Palo Alto") & liked_by("Kathy")))

🔎 Text-to-SQL is now usually done by prompting an LLM with the database schema (Spider and BIRD are standard benchmarks), but the target is the same executable meaning, and **execution accuracy** (does the generated query return the right answer?) is the natural metric: a direct descendant of model-theoretic semantics.

### 5.3 Language processing in humans, and neural composition
There is evidence that, to get the meaning of a sentence, **humans also perform hierarchical computations**, mostly following projective bottom-up trees: structure dependence in children's grammar formation (Crain & Nakayama 1987), cortical representation of constituent structure (Pallier et al. 2011), cortical tracking of hierarchical structure in connected speech (Ding et al. 2016), and syntax in human EEG via beam search (Hale et al. 2018). **Do neural models provide suitable meaning (composition) functions?** (L16's TreeRNNs and RNTNs were an explicit attempt; Transformers learn implicit, soft structure, and L4's structural probes found trees inside them.)

---
## 6. Meaning as use, and "meaning is gradient"

**Wittgenstein, *Philosophical Investigations*:** *"You say: the point isn't the word, but its meaning, and you think of the meaning as a thing of the same kind as the word, though also different from the word. Here the word, there the meaning. The money, and the cow that you can buy with it. (But contrast: money, and its use.)"* Meaning as **use**, not as a separate object.

**Is that semantics/meaning?** Some people don't accept distributional meaning as a theory of semantics (e.g., Bender & Koller 2020, whose "octopus" thought experiment argues that a system trained only on form cannot learn meaning). Manning's position:
* **Meaning arises from the connection of words to other things.** The "real world" is in some sense privileged, but it's not the only grounding of meaning: there are also virtual worlds, and human language itself.
* **Meaning is gradient:** how well do you understand a word or phrase?

**What is the meaning of *shehnai*?** (Manning, *Dædalus* 2022) (1) You may have seen or held one: a classic grounded meaning. (2) I can at least show you a picture. (3) Surely the meaning would be richer if you had heard one played. (4) If you have never seen, felt, or heard a shehnai, but someone tells you it's a **traditional Indian instrument, a bit like an oboe**, then surely the word has *some* meaning for you: connections to India, to reed instruments, to playing music. (5) If they add that it has **holes like a recorder, but multiple reeds and a flared end more like an oboe**, you have more network connections, more "meaning." And from **contexts of use alone** (the slides quote a novel in which shehnai players play at a wedding), you still learn it's a pipe-like musical instrument, along with musical and cultural connections that someone who has merely held one might lack.

We can watch distributional meaning grow, using a tiny corpus (sentences written for this demo) and adding contexts of a new word one at a time:

In [ ]:
corpus = '''the oboe is a reed instrument played by blowing air through it
the musician played a slow melody on the oboe at the concert
she practiced the flute every evening and played a melody for the guests
the clarinet is a reed instrument with holes and keys
he played the drum loudly at the festival with the band
the drummer kept the rhythm with a drum at the concert
the car drove down the road and parked near the house
she drove her car to the market in the morning
the bus stopped near the station and the passengers walked home
we cooked rice and lentils for dinner with the family
the soup was hot and we ate it with bread at dinner'''.split("\n")
new_contexts = ["at the wedding the musicians played the shehnai and a slow melody filled the hall",
                "the shehnai is a reed instrument from india a bit like the oboe",
                "the shehnai has holes and a flared end and it is played by blowing air through it"]

def cooc_vectors(sentences, window=4):
    vocab = sorted({w for s in sentences for w in s.split()}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for s in sentences:
        ws = s.split()
        for i, w in enumerate(ws):
            for j in range(max(0, i - window), min(len(ws), i + window + 1)):
                if j != i:
                    M[idx[w], idx[ws[j]]] += 1
    tot = M.sum(); pw = M.sum(1, keepdims=True) / tot; pc = M.sum(0, keepdims=True) / tot
    with np.errstate(divide="ignore"):
        ppmi = np.maximum(np.log((M / tot) / (pw * pc)), 0)               # PPMI (L2 §4.5)
    return ppmi, idx

probe = ["oboe", "clarinet", "flute", "drum", "car", "bus", "soup"]
print(f"{'contexts of shehnai':22s}" + "".join(f"{w:>10s}" for w in probe))
for k in range(1, len(new_contexts) + 1):
    M, idx = cooc_vectors(corpus + new_contexts[:k])
    v = M[idx["shehnai"]]
    sims = [v @ M[idx[w]] / (np.linalg.norm(v) * np.linalg.norm(M[idx[w]]) + 1e-12) for w in probe]
    print(f"{k} context(s){'':12s}" + "".join(f"{s_:10.2f}" for s_ in sims))

With one context of use, *shehnai* already sits closer to the instruments than to cars or soup (it occurs with *played*, *musicians*, *melody*). The second context (*a reed instrument … like the oboe*) ties it sharply to the reed instruments. The third adds new connections (*holes*, *a flared end*) that our tiny corpus barely knows, so the cosines dip, but the ranking stays: reed instruments, then other instruments, then everything else. A single similarity number is a crude summary. On this view, meaning is a **network of connections that can be richer or poorer**, not all-or-nothing.

---
## 7. The future risks of AI in the world

**Are we all going to lose our jobs?** *"In the past, new industries hired far more people than those they put out of business. But this is not true of many of today's new industries…"* (*Time* magazine, **1961**). The worry is old; so far, technology has transformed work more than eliminated it, but the distribution of gains is the question.

**Will almost all the money go to 5–10 enormous technology giants?**

**Should we be afraid of an imminent "singularity"** (machines with artificial general intelligence beyond human level)? Would it threaten human survival? The AI existential-risk (x-risk) debate has exploded into the mainstream (*State of AI Report*). These arguments have many critics, who question the logic and sometimes the motivations of their proponents:
* **François Chollet** (creator of Keras) argues no existing AI technique represents an extinction risk, even extrapolating capabilities via scaling laws; most arguments, he says, boil down to "this is a new type of technology … it could happen."
* **Joelle Pineau** (Meta AI) warned that "when you put an infinite cost on x-risk, you can't have any rational discussion about other outcomes."
* **Timnit Gebru** (DAIR) argues that x-risk talk distracts from **immediate harms** of deployed systems: biased systems, worker exploitation, copyright violation, disinformation, and growing concentration of power and regulatory capture by the leading AI companies.

**There are always hidden costs to AI development:** human labor (e.g., data annotation) and environmental costs (L19 §8).

**NLP harms:** models can **generate offensive content** (the slides show a completion that associates Muslims with violence, cf. Abid et al. 2021), **untruthful content** (an invented history of Stanford's founding), and **enable disinformation** (a fluent climate-denial claim).

**Misinformation:** if models can learn to reason better about real-world text, can they more convincingly **fake knowledge** too? Can they be more **persuasive** with incorrect information, perhaps personalized, in politics or elsewhere? The literature suggests **yes, they can** (debated: Simon et al. 2023 vs. Tomz et al. 2024), and multiple studies suggest humans are **more likely to believe AI-generated disinformation**; AI-generated political propaganda already works (Tomz et al. 2024, using GPT-3-era models). AI-generated audio and visual content may be even more persuasive and viral. **Solutions so far:** labeling AI-generated content, tracking real human-generated content (**watermarking**), and educating users.

**Worry about what people and organizations with power will use AI to do.** The lecture closes with Carl Sagan's warning (*The Demon-Haunted World*, 1995) of a future where awesome technological powers are in the hands of a very few, and the public loses the ability to set its own agendas or knowledgeably question those in authority, *"unable to distinguish between what feels good and what's true."*

### 7.1 🔎 Watermarking LLM output (Kirchenbauer et al., ICML 2023)
One concrete technical response to AI-generated disinformation. At each step, use the **previous token** to seed a random split of the vocabulary into a **green list** (fraction $\gamma$) and a red list, and add a small bias $\delta$ to green tokens' logits before sampling. The text reads normally, but contains more green tokens than chance. **Detection needs no model access**, only the hashing scheme: count green tokens $|s|_G$ in $T$ tokens and compute
$$z = \frac{|s|_G - \gamma T}{\sqrt{T\gamma(1-\gamma)}}$$
Under the null hypothesis (human text), $z\approx N(0,1)$; watermarked text gives large $z$.

In [ ]:
r = np.random.default_rng(0)
VOCAB, GAMMA, DELTA = 1000, 0.25, 2.0
LM = r.normal(0, 2.0, (VOCAB, VOCAB))                    # a toy LM: next-token logits depend on the previous token

def green_list(prev):
    return np.random.default_rng(prev * 7919 + 17).random(VOCAB) < GAMMA     # seeded by the previous token

def generate(n, watermark, r_, start=0):
    toks = [start]
    for _ in range(n):
        logits = LM[toks[-1]].copy()
        if watermark:
            logits[green_list(toks[-1])] += DELTA          # nudge toward green tokens
        toks.append(int(r_.choice(VOCAB, p=softmax(logits))))
    return toks

def z_score(toks):
    hits = sum(green_list(a)[b] for a, b in zip(toks, toks[1:])); T = len(toks) - 1
    return (hits - GAMMA * T) / math.sqrt(T * GAMMA * (1 - GAMMA))

def perplexity_under_lm(toks):
    lp = [math.log(softmax(LM[a])[b]) for a, b in zip(toks, toks[1:])]
    return math.exp(-np.mean(lp))

for T in [25, 50, 200]:
    zs_h = [z_score(generate(T, False, r, start=int(r.integers(VOCAB)))) for _ in range(30)]
    zs_w = [z_score(generate(T, True, r, start=int(r.integers(VOCAB)))) for _ in range(30)]
    print(f"{T:3d} tokens: z (unwatermarked) mean {np.mean(zs_h):+.2f}   z (watermarked) mean {np.mean(zs_w):+.2f}   "
          f"detected at z > 4: {np.mean(np.array(zs_w) > 4):.0%} of watermarked, {np.mean(np.array(zs_h) > 4):.0%} false positives")

text_w = generate(200, True, r); text_h = generate(200, False, r)
print(f"\nquality cost: perplexity under the original LM, unwatermarked {perplexity_under_lm(text_h):.1f} vs watermarked {perplexity_under_lm(text_w):.1f}")
for frac in [0.1, 0.3, 0.5]:                              # robustness: an attacker replaces some tokens at random
    edited = [t if r.random() > frac else int(r.integers(VOCAB)) for t in text_w]
    print(f"after randomly replacing {frac:.0%} of tokens: z = {z_score(edited):+.2f}")

Detection becomes reliable within a few dozen tokens, false positives stay at zero at a high threshold, and random edits weaken but don't erase the signal (each replaced token breaks the green list for itself and the next token). The costs: a small increase in perplexity (the text is nudged away from the model's preferred choices), weak signals on **low-entropy** text (where there's only one sensible next token), and vulnerability to **paraphrasing** attacks. Watermarking only works if the generator cooperates, which is why it's paired with provenance standards for authentic content (e.g., C2PA) and with user education.

---
## 8. Summary

| Topic | Takeaway |
|---|---|
| Major ideas | Dense distributed representations; depth; sequence models and efficiency; language modeling + pretraining |
| Generalization | Models can learn **shortcuts** that fit the training distribution and fail beyond it (e.g., at longer lengths) |
| LLMs now | Large productivity gains on some tasks (esp. for lower performers), poor on others (creative writing, out-of-envelope tasks); hype vs. limitations |
| Symbolic vs. neural | Language is a symbolic system; its processor need not be; neural models scale; perceptron (1958) → XOR limit → hidden layers |
| Linguistics | The right tools for goals, analysis, and evaluation of NLP: compositionality, systematic generalization, reference |
| Formal semantics | Montague: syntax→semantics homomorphism; λ-calculus composition; presupposition; semantic parsing to executable queries |
| Distributional meaning | Meaning as use (Wittgenstein); meaning arises from connections, and is **gradient** (*shehnai*) |
| Risks | Jobs, concentration of power, x-risk debate vs. immediate harms, offensive/untruthful content, persuasive misinformation |
| 🔎 Watermarking | Green-list bias at generation; z-test detection; costs and attacks |

### Exercises
1. Add *every*, *some*, and *no* to the lexicon of §5.1 as generalized quantifiers ($\lambda P.\lambda Q.\ \forall x(P(x)\rightarrow Q(x))$, etc.) and interpret *every red apple is on the table*.
2. Extend the semantic parser in §5.2 to handle *Which red cars in Palo Alto does Kathy like?* (return the set, not the count) and *Does Kathy like every red car?*
3. Run the perceptron on data with margins 0.05, 0.1, 0.3, and check the mistake counts against $(R/\gamma)^2$.
4. In §7.1, measure detection on **low-entropy** text: make the toy LM peakier (multiply `LM` by 5). What happens to $z$ and why?
5. Implement a **paraphrase attack**: replace each token with the LM's second-most-likely token given the previous one. How much does $z$ drop compared with random replacement?

### References
* Liu et al. 2023, *Transformers Learn Shortcuts to Automata*; Karpathy, Johnson & Fei-Fei 2016 (*Visualizing and Understanding Recurrent Networks*).
* Dell'Acqua et al. 2023 (BCG field experiment); Chakrabarty et al. 2023 (LLM creative writing); Brynjolfsson, Li & Raymond 2023 (NBER customer support).
* Rosenblatt 1958; Novikoff 1962; Minsky & Papert 1969.
* Montague 1970, *Universal Grammar*; Partee 1996; Fodor & Pylyshyn 1988; Brooks & Tomasello 1999.
* Zettlemoyer & Collins 2005; Liang, Jordan & Klein 2013; Pallier et al. 2011; Ding et al. 2016; Hale et al. 2018.
* Wittgenstein 1953, *Philosophical Investigations*; Bender & Koller 2020 (*Climbing towards NLU*); Manning 2022, *Human Language Understanding & Reasoning* (Dædalus).
* Abid, Farooqi & Zou 2021 (anti-Muslim bias in GPT-3); Tomz et al. 2024; Kirchenbauer et al. 2023 (*A Watermark for Large Language Models*).